<a href="https://colab.research.google.com/github/Henriquelcs/Calistenia-em-casa/blob/main/OPPORTUNITY_RADAR_CELULA_39C_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# CÉLULA 1 — MANUTENÇÃO DO PRODUTO
# OR-003.2 — PRESERVAR EVIDÊNCIA TEXTUAL ATÉ O ASSESSMENT
# VERSÃO: 2026-08-25.1
# =============================================================================

import base64
import os
import subprocess
import sys
from pathlib import Path

from google.colab import userdata


# =============================================================================
# IDENTIDADE
# =============================================================================

CELL_NAME = "CÉLULA 1 — MANUTENÇÃO DO PRODUTO"
INCIDENT = "OR-003.2"
VERSION = "2026-08-25.1"

PROJECT_DIR = Path("/content/opportunity-radar")

DATA_ACCESS_FILE = PROJECT_DIR / "src/dashboard/data_access.py"
TEST_FILE = PROJECT_DIR / "tests/test_dashboard_data_access.py"

COMMIT_MESSAGE = (
    "fix: preserve opportunity evidence through dashboard data pipeline"
)


print("=" * 80)
print(CELL_NAME)
print("OR-003.2 — PRESERVAR EVIDÊNCIA TEXTUAL ATÉ O ASSESSMENT")
print(f"VERSÃO: {VERSION}")
print("=" * 80)


# =============================================================================
# HELPERS
# =============================================================================

def command_output(command, cwd=PROJECT_DIR):
    return subprocess.check_output(
        list(map(str, command)),
        cwd=str(cwd),
        text=True,
    ).strip()


def run_checked(command, cwd=PROJECT_DIR, env=None):
    print("$", " ".join(map(str, command)), flush=True)

    result = subprocess.run(
        list(map(str, command)),
        cwd=str(cwd),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            "Comando falhou:\n"
            + " ".join(map(str, command))
        )

    return result


def run_diagnostic(command, title, cwd=PROJECT_DIR):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    print("$", " ".join(map(str, command)), flush=True)

    result = subprocess.run(
        list(map(str, command)),
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    print(result.stdout)
    print("=" * 80)

    if result.returncode != 0:
        raise RuntimeError(
            f"{title} FALHOU.\n"
            "Nenhum commit ou push foi realizado."
        )

    print(f"✅ {title}: APROVADO")

    return result


# =============================================================================
# 1. VALIDAR REPOSITÓRIO
# =============================================================================

if not (PROJECT_DIR / ".git").exists():
    raise RuntimeError(
        f"Repositório não encontrado: {PROJECT_DIR}"
    )


status_before = subprocess.check_output(
    [
        "git",
        "status",
        "--porcelain",
    ],
    cwd=str(PROJECT_DIR),
    text=True,
)

if status_before.strip():
    raise RuntimeError(
        "Existem alterações locais antes do OR-003.2.\n"
        "Não vou misturar incidentes:\n\n"
        + status_before
    )


run_checked(
    [
        "git",
        "pull",
        "--ff-only",
        "origin",
        "main",
    ]
)


base_sha = command_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)

print(f"[OR-003.2] base_sha={base_sha}")


# =============================================================================
# 2. VALIDAR BASE ATUAL
# =============================================================================

data_text = DATA_ACCESS_FILE.read_text(
    encoding="utf-8"
)

required_markers = (
    "def normalize_opportunities(",
    "def load_radar_data(",
    "def enrich_opportunities(",
)

missing = [
    marker
    for marker in required_markers
    if marker not in data_text
]

if missing:
    raise RuntimeError(
        "Estrutura esperada do data_access.py não encontrada:\n"
        + "\n".join(
            f"- {marker}"
            for marker in missing
        )
    )

print("✅ Base data_access confirmada.")


# =============================================================================
# 3. PRESERVAR CAMPOS TEXTUAIS EM normalize_opportunities
# =============================================================================

old_columns = '''    columns = [
        "database_file",
        "database_path",
        "opportunity_id",
        "source",
        "title",
        "url",
        "score",
        "level",
        "pain_categories",
        "created_at",
    ]
'''

new_columns = '''    columns = [
        "database_file",
        "database_path",
        "opportunity_id",
        "source",
        "title",
        "description",
        "body",
        "content",
        "summary",
        "url",
        "score",
        "level",
        "pain_categories",
        "created_at",
    ]
'''

if new_columns in data_text:

    print(
        "[OR-003.2] colunas textuais já estão normalizadas."
    )

elif old_columns in data_text:

    data_text = data_text.replace(
        old_columns,
        new_columns,
        1,
    )

    print(
        "[OR-003.2] colunas description/body/content/summary adicionadas."
    )

else:
    raise RuntimeError(
        "OR-003.2: lista de colunas de normalize_opportunities "
        "não foi localizada."
    )


# =============================================================================
# 4. LER OS CAMPOS DO BANCO
# =============================================================================

old_title_block = '''    result["title"] = _string_series(
        _series_or_default(frame, ["title", "name"])
    )
    result["url"] = _string_series(
'''

new_title_block = '''    result["title"] = _string_series(
        _series_or_default(frame, ["title", "name"])
    )

    # OR-003.2
    #
    # O Runner já persiste description no banco operacional.
    # Estes campos precisam sobreviver até o assessment/dashboard.
    result["description"] = _string_series(
        _series_or_default(
            frame,
            [
                "description",
                "details",
                "text",
            ],
        )
    )

    result["body"] = _string_series(
        _series_or_default(
            frame,
            [
                "body",
            ],
        )
    )

    result["content"] = _string_series(
        _series_or_default(
            frame,
            [
                "content",
            ],
        )
    )

    result["summary"] = _string_series(
        _series_or_default(
            frame,
            [
                "summary",
                "excerpt",
            ],
        )
    )

    result["url"] = _string_series(
'''

if new_title_block in data_text:

    print(
        "[OR-003.2] leitura da evidência textual já existe."
    )

elif old_title_block in data_text:

    data_text = data_text.replace(
        old_title_block,
        new_title_block,
        1,
    )

    print(
        "[OR-003.2] evidência textual agora é preservada."
    )

else:
    raise RuntimeError(
        "OR-003.2: bloco title/url não localizado."
    )


DATA_ACCESS_FILE.write_text(
    data_text,
    encoding="utf-8",
)


# =============================================================================
# 5. TESTE DE INTEGRAÇÃO REAL
# =============================================================================

test_text = TEST_FILE.read_text(
    encoding="utf-8"
)


# -------------------------------------------------------------------------
# 5.1 Importar assess_frame
# -------------------------------------------------------------------------

assessment_import = (
    "from src.product.assessment import assess_frame"
)

if assessment_import not in test_text:

    anchor = (
        "from src.dashboard.data_access "
        "import discover_databases, load_radar_data\n"
    )

    if anchor not in test_text:
        raise RuntimeError(
            "OR-003.2: import de data_access não localizado."
        )

    test_text = test_text.replace(
        anchor,
        anchor + assessment_import + "\n",
        1,
    )


# -------------------------------------------------------------------------
# 5.2 Teste atravessando banco → data_access → assessment
# -------------------------------------------------------------------------

test_marker = (
    "def test_or0032_persisted_description_reaches_decision_assessment"
)

if test_marker not in test_text:

    integration_test = r'''


def test_or0032_persisted_description_reaches_decision_assessment(
    tmp_path: Path,
) -> None:
    """
    Regressão OR-003.2.

    Simula exatamente o caminho real:

    SQLite
    -> load_radar_data()
    -> normalize_opportunities()
    -> assess_frame()
    -> decision_eligibility

    O corpo técnico da oportunidade não pode desaparecer
    entre banco e dashboard.
    """
    database = tmp_path / "or0032.db"

    fleet_description = (
        "This issue coordinates exact-subject reconciliation of "
        "independently developed provider-capacity and runtime-control "
        "implementations. Required comparison dimensions include "
        "quota-domain separation, full-child lifetime lease, "
        "PID/session/epoch fencing, provider-neutral normalization, "
        "telemetry, closed-by-default gate, security review, "
        "hosted checks and a single canary."
    )

    with sqlite3.connect(database) as connection:
        connection.executescript(
            """
            CREATE TABLE opportunities (
                id INTEGER PRIMARY KEY,
                source TEXT,
                title TEXT,
                description TEXT,
                url TEXT UNIQUE,
                opportunity_score REAL,
                opportunity_level TEXT,
                pain_categories TEXT
            );

            CREATE TABLE query_expansion_runs (
                id INTEGER PRIMARY KEY,
                original_query TEXT,
                status TEXT,
                variation_count INTEGER,
                successful_variations INTEGER,
                failed_variations INTEGER,
                total_matches INTEGER,
                unique_opportunities INTEGER,
                duplicate_matches INTEGER
            );

            CREATE TABLE opportunity_query_matches (
                id INTEGER PRIMARY KEY,
                expansion_run_id INTEGER,
                variation_id INTEGER,
                opportunity_id TEXT,
                opportunity_url TEXT,
                source TEXT,
                title TEXT,
                original_query TEXT,
                matched_query TEXT,
                first_seen_at TEXT
            );
            """
        )

        connection.execute(
            """
            INSERT INTO opportunities (
                id,
                source,
                title,
                description,
                url,
                opportunity_score,
                opportunity_level,
                pain_categories
            )
            VALUES (
                1,
                'github',
                'Fleet reconciliation: universal provider-control implementations',
                ?,
                'https://github.com/layibabalola/softwarefactory-fleet-doctrine/issues/4',
                56.0,
                'medium',
                'failure_or_blocker | automation_demand'
            )
            """,
            (fleet_description,),
        )

        connection.execute(
            """
            INSERT INTO query_expansion_runs (
                id,
                original_query,
                status,
                variation_count,
                successful_variations,
                failed_variations,
                total_matches,
                unique_opportunities,
                duplicate_matches
            )
            VALUES (
                1,
                'manual reconciliation spreadsheet operations',
                'SUCCESS',
                1,
                1,
                0,
                1,
                1,
                0
            )
            """
        )

        connection.execute(
            """
            INSERT INTO opportunity_query_matches (
                id,
                expansion_run_id,
                variation_id,
                opportunity_id,
                opportunity_url,
                source,
                title,
                original_query,
                matched_query,
                first_seen_at
            )
            VALUES (
                1,
                1,
                1,
                '1',
                'https://github.com/layibabalola/softwarefactory-fleet-doctrine/issues/4',
                'github',
                'Fleet reconciliation: universal provider-control implementations',
                'manual reconciliation spreadsheet operations',
                'manual reconciliation spreadsheet operations',
                '2026-08-25T18:00:00Z'
            )
            """
        )

    dataset = load_radar_data(tmp_path)

    assert len(dataset.opportunities) == 1

    loaded = dataset.opportunities.iloc[0]

    # A evidência precisa sobreviver ao data_access.
    assert loaded["description"] == fleet_description

    assessed = assess_frame(
        dataset.opportunities
    )

    assert len(assessed) == 1

    row = assessed.iloc[0]

    # O OR-003.1 agora recebe os mesmos dados da execução real.
    assert row["decision_eligible"] == False

    assert (
        row["decision_eligibility_reason"]
        == "technical_governance_artifact"
    )


def test_or0032_operational_description_survives_and_remains_eligible(
    tmp_path: Path,
) -> None:
    """
    Controle positivo:
    preservar description não pode bloquear uma dor operacional real.
    """
    database = tmp_path / "or0032_positive.db"

    operational_description = (
        "Our backoffice team exports two CSV files every morning, "
        "compares records manually in Excel and copy-pastes mismatches "
        "between systems. The reconciliation takes hours every day."
    )

    with sqlite3.connect(database) as connection:
        connection.executescript(
            """
            CREATE TABLE opportunities (
                id INTEGER PRIMARY KEY,
                source TEXT,
                title TEXT,
                description TEXT,
                url TEXT UNIQUE,
                opportunity_score REAL,
                opportunity_level TEXT,
                pain_categories TEXT
            );

            CREATE TABLE query_expansion_runs (
                id INTEGER PRIMARY KEY,
                original_query TEXT,
                status TEXT,
                variation_count INTEGER,
                successful_variations INTEGER,
                failed_variations INTEGER,
                total_matches INTEGER,
                unique_opportunities INTEGER,
                duplicate_matches INTEGER
            );

            CREATE TABLE opportunity_query_matches (
                id INTEGER PRIMARY KEY,
                expansion_run_id INTEGER,
                variation_id INTEGER,
                opportunity_id TEXT,
                opportunity_url TEXT,
                source TEXT,
                title TEXT,
                original_query TEXT,
                matched_query TEXT,
                first_seen_at TEXT
            );
            """
        )

        connection.execute(
            """
            INSERT INTO opportunities (
                id,
                source,
                title,
                description,
                url,
                opportunity_score,
                opportunity_level,
                pain_categories
            )
            VALUES (
                1,
                'github',
                'Manual spreadsheet reconciliation takes hours',
                ?,
                'https://example.com/operational-reconciliation',
                65.0,
                'medium',
                'manual_work | repetition | workflow_gap'
            )
            """,
            (operational_description,),
        )

        connection.execute(
            """
            INSERT INTO query_expansion_runs (
                id,
                original_query,
                status,
                variation_count,
                successful_variations,
                failed_variations,
                total_matches,
                unique_opportunities,
                duplicate_matches
            )
            VALUES (
                1,
                'manual reconciliation spreadsheet operations',
                'SUCCESS',
                1,
                1,
                0,
                1,
                1,
                0
            )
            """
        )

        connection.execute(
            """
            INSERT INTO opportunity_query_matches (
                id,
                expansion_run_id,
                variation_id,
                opportunity_id,
                opportunity_url,
                source,
                title,
                original_query,
                matched_query,
                first_seen_at
            )
            VALUES (
                1,
                1,
                1,
                '1',
                'https://example.com/operational-reconciliation',
                'github',
                'Manual spreadsheet reconciliation takes hours',
                'manual reconciliation spreadsheet operations',
                'manual reconciliation spreadsheet operations',
                '2026-08-25T18:00:00Z'
            )
            """
        )

    dataset = load_radar_data(tmp_path)

    loaded = dataset.opportunities.iloc[0]

    assert loaded["description"] == operational_description

    row = assess_frame(
        dataset.opportunities
    ).iloc[0]

    assert row["decision_eligible"] == True

    assert (
        row["decision_eligibility_reason"]
        == "problem_evidence_present"
    )
'''

    test_text += integration_test

    print(
        "[OR-003.2] testes de integração adicionados."
    )

else:
    print(
        "[OR-003.2] testes de integração já existem."
    )


TEST_FILE.write_text(
    test_text,
    encoding="utf-8",
)


# =============================================================================
# 6. MOSTRAR DIFF
# =============================================================================

print("\n" + "=" * 80)
print("OR-003.2 — DIFF")
print("=" * 80)

diff_result = subprocess.run(
    [
        "git",
        "diff",
        "--",
        "src/dashboard/data_access.py",
        "tests/test_dashboard_data_access.py",
    ],
    cwd=str(PROJECT_DIR),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(diff_result.stdout)
print("=" * 80)


# =============================================================================
# 7. COMPILEALL
# =============================================================================

run_diagnostic(
    [
        sys.executable,
        "-m",
        "compileall",
        "-q",
        "src",
        "scripts",
        "tests",
    ],
    "OR-003.2 — COMPILEALL",
)


# =============================================================================
# 8. TESTE ESPECÍFICO DE INTEGRAÇÃO
# =============================================================================

run_diagnostic(
    [
        sys.executable,
        "-m",
        "pytest",
        "-vv",
        "--tb=long",
        "tests/test_dashboard_data_access.py",
        "-k",
        "or0032",
    ],
    "OR-003.2 — TESTES DE INTEGRAÇÃO",
)


# =============================================================================
# 9. DATA ACCESS COMPLETO
# =============================================================================

run_diagnostic(
    [
        sys.executable,
        "-m",
        "pytest",
        "-vv",
        "--tb=long",
        "tests/test_dashboard_data_access.py",
    ],
    "OR-003.2 — DATA ACCESS COMPLETO",
)


# =============================================================================
# 10. ASSESSMENT COMPLETO
# =============================================================================

run_diagnostic(
    [
        sys.executable,
        "-m",
        "pytest",
        "-vv",
        "--tb=long",
        "tests/test_product_assessment.py",
    ],
    "OR-003.2 — ASSESSMENT COMPLETO",
)


# =============================================================================
# 11. SUÍTE COMPLETA
# =============================================================================

run_diagnostic(
    [
        sys.executable,
        "-m",
        "pytest",
        "-x",
        "-vv",
        "--tb=long",
    ],
    "OR-003.2 — SUÍTE COMPLETA",
)


# =============================================================================
# 12. DIFF CHECK
# =============================================================================

run_diagnostic(
    [
        "git",
        "diff",
        "--check",
    ],
    "OR-003.2 — DIFF CHECK",
)


# =============================================================================
# 13. AUTENTICAÇÃO GITHUB
# =============================================================================

github_token = None
github_secret_name = None

for secret_name in (
    "GITHUB_TOKEN",
    "GH_TOKEN",
    "GITHUB_PAT",
):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None

    if value:
        github_token = str(value)
        github_secret_name = secret_name
        break


if not github_token:
    raise RuntimeError(
        "Todos os testes passaram, mas token GitHub "
        "não foi encontrado."
    )


print(
    f"[AUTH] usando segredo {github_secret_name}"
)


# =============================================================================
# 14. CONFIGURAÇÃO GIT
# =============================================================================

run_checked(
    [
        "git",
        "config",
        "user.name",
        "Henrique Luiz Costa da Silva",
    ]
)

run_checked(
    [
        "git",
        "config",
        "user.email",
        "Henriquelcs@users.noreply.github.com",
    ]
)


# =============================================================================
# 15. STAGE CONTROLADO
# =============================================================================

allowed_files = {
    "src/dashboard/data_access.py",
    "tests/test_dashboard_data_access.py",
}


run_checked(
    [
        "git",
        "add",
        "src/dashboard/data_access.py",
        "tests/test_dashboard_data_access.py",
    ]
)


staged_files = {
    path.strip()
    for path in subprocess.check_output(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ],
        cwd=str(PROJECT_DIR),
        text=True,
    ).splitlines()
    if path.strip()
}


unexpected_staged = staged_files - allowed_files

if unexpected_staged:
    raise RuntimeError(
        "Arquivos inesperados no stage:\n"
        + "\n".join(
            f"- {path}"
            for path in sorted(unexpected_staged)
        )
    )


# =============================================================================
# 16. COMMIT IDPOTENTE
# =============================================================================

has_changes = subprocess.run(
    [
        "git",
        "diff",
        "--cached",
        "--quiet",
    ],
    cwd=str(PROJECT_DIR),
)


if has_changes.returncode == 0:

    print()
    print(
        "✅ OR-003.2 já estava aplicado."
    )
    print(
        "✅ Nenhum novo commit necessário."
    )

    local_sha = command_output(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

else:

    print("[OR-003.2] Arquivos no stage:")

    for path in sorted(staged_files):
        print(f"  - {path}")

    run_checked(
        [
            "git",
            "commit",
            "-m",
            COMMIT_MESSAGE,
        ]
    )

    local_sha = command_output(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

    print(
        f"[COMMIT] local_sha={local_sha}"
    )


# =============================================================================
# 17. PUSH
# =============================================================================

auth_value = base64.b64encode(
    f"x-access-token:{github_token}".encode(
        "utf-8"
    )
).decode(
    "ascii"
)

push_env = os.environ.copy()

push_env.update(
    {
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0":
            "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0":
            f"AUTHORIZATION: basic {auth_value}",
    }
)


run_checked(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    env=push_env,
)


# =============================================================================
# 18. CONFIRMAR REMOTO
# =============================================================================

run_checked(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_sha = command_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)

remote_sha = command_output(
    [
        "git",
        "rev-parse",
        "origin/main",
    ]
)


if local_sha != remote_sha:
    raise RuntimeError(
        "Push OR-003.2 não confirmado.\n"
        f"LOCAL : {local_sha}\n"
        f"REMOTO: {remote_sha}"
    )


# =============================================================================
# 19. GIT LIMPO
# =============================================================================

final_status = subprocess.check_output(
    [
        "git",
        "status",
        "--porcelain",
    ],
    cwd=str(PROJECT_DIR),
    text=True,
)


if final_status.strip():
    raise RuntimeError(
        "OR-003.2 terminou com Git sujo:\n"
        + final_status
    )


# =============================================================================
# 20. ENTREGA
# =============================================================================

print("\n" + "=" * 80)
print("✅ CÉLULA 1 CONCLUÍDA")
print(f"✅ VERSÃO: {VERSION}")
print("✅ INCIDENTE: OR-003.2")
print("✅ RUNNER: DESCRIPTION JÁ EXISTIA NO BANCO")
print("✅ DATA ACCESS: DESCRIPTION PRESERVADA")
print("✅ BODY/CONTENT/SUMMARY: PRESERVADOS QUANDO EXISTIREM")
print("✅ BANCO → DATA_ACCESS → ASSESSMENT: TESTADO")
print("✅ FLEET RECONCILIATION: TESTE REAL DE INTEGRAÇÃO")
print("✅ RECONCILIAÇÃO OPERACIONAL: CONTROLE POSITIVO")
print("✅ DATA ACCESS COMPLETO: APROVADO")
print("✅ ASSESSMENT COMPLETO: APROVADO")
print("✅ SUÍTE COMPLETA: APROVADA")
print(f"✅ COMMIT LOCAL: {local_sha}")
print(f"✅ COMMIT REMOTO: {remote_sha}")
print("✅ GIT: LIMPO")
print("=" * 80)

CÉLULA 1 — MANUTENÇÃO DO PRODUTO
OR-003.2 — PRESERVAR EVIDÊNCIA TEXTUAL ATÉ O ASSESSMENT
VERSÃO: 2026-08-25.1
$ git pull --ff-only origin main
From https://github.com/Henriquelcs/opportunity-radar
 * branch            main       -> FETCH_HEAD
Already up to date.

[OR-003.2] base_sha=3f13e41813fcceb0e7f481de83f49a63d65e7784
✅ Base data_access confirmada.
[OR-003.2] colunas description/body/content/summary adicionadas.
[OR-003.2] evidência textual agora é preservada.
[OR-003.2] testes de integração adicionados.

OR-003.2 — DIFF
diff --git a/src/dashboard/data_access.py b/src/dashboard/data_access.py
index e3d5e2d..3a140e8 100644
--- a/src/dashboard/data_access.py
+++ b/src/dashboard/data_access.py
@@ -131,6 +131,10 @@ def normalize_opportunities(frame: pd.DataFrame) -> pd.DataFrame:
         "opportunity_id",
         "source",
         "title",
+        "description",
+        "body",
+        "content",
+        "summary",
         "url",
         "score",
         "level",
@@ -156,6 +

In [ ]:
# =============================================================================
# CÉLULA 2 — OPERAÇÃO + DASHBOARD
# VERSÃO: 2026-08-25.4
# =============================================================================

import os
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

print("=" * 80)
print("CÉLULA 2 — OPERAÇÃO + DASHBOARD")
print("VERSÃO: 2026-08-25.3")
print("=" * 80)

REPOSITORY_URL = "https://github.com/Henriquelcs/opportunity-radar.git"
PROJECT_DIR = Path("/content/opportunity-radar")
RUNTIME_DIR = Path("/content/opportunity-radar-runtime")
URL_FILE = RUNTIME_DIR / "dashboard_url.txt"

QUERY = "manual reconciliation spreadsheet operations"
LIMIT = "30"
MINIMUM_SCORE = "0"


def run(command, cwd=None):
    print("$", " ".join(map(str, command)), flush=True)

    subprocess.run(
        list(map(str, command)),
        cwd=str(cwd) if cwd else None,
        check=True,
    )


# -------------------------------------------------------------------------
# 1. Secrets
# -------------------------------------------------------------------------

for secret_name in (
    "GITHUB_TOKEN",
    "GH_TOKEN",
    "GITHUB_PAT",
    "STACKEXCHANGE_KEY",
    "DEVTO_API_KEY",
):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None

    if value:
        os.environ[secret_name] = str(value)


# -------------------------------------------------------------------------
# 2. Repositório
# -------------------------------------------------------------------------

if not (PROJECT_DIR / ".git").exists():
    run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPOSITORY_URL,
            PROJECT_DIR,
        ]
    )
else:
    run(
        ["git", "pull", "--ff-only", "origin", "main"],
        cwd=PROJECT_DIR,
    )


# -------------------------------------------------------------------------
# 3. Setup
# -------------------------------------------------------------------------

run(
    [
        sys.executable,
        "scripts/run_colab.py",
        "--mode",
        "setup",
    ],
    cwd=PROJECT_DIR,
)


# -------------------------------------------------------------------------
# 4. Coleta
# -------------------------------------------------------------------------

run(
    [
        sys.executable,
        "scripts/run_colab.py",
        "--mode",
        "collect",
        "--query",
        QUERY,
        "--limit",
        LIMIT,
        "--minimum-score",
        MINIMUM_SCORE,
    ],
    cwd=PROJECT_DIR,
)


# -------------------------------------------------------------------------
# 5. Remover URL antiga
# -------------------------------------------------------------------------

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

if URL_FILE.exists():
    URL_FILE.unlink()


# -------------------------------------------------------------------------
# 6. Subir dashboard + túnel
# -------------------------------------------------------------------------

print("\n[DASHBOARD] Iniciando dashboard pública...")

run(
    [
        sys.executable,
        "scripts/run_colab.py",
        "--mode",
        "dashboard",
        "--dashboard-wait-seconds",
        "90",
    ],
    cwd=PROJECT_DIR,
)


# -------------------------------------------------------------------------
# 7. Validar URL gerada
# -------------------------------------------------------------------------

if not URL_FILE.exists():
    raise RuntimeError(
        "Dashboard terminou sem gerar dashboard_url.txt."
    )

dashboard_url = URL_FILE.read_text(
    encoding="utf-8"
).strip()

if not dashboard_url.startswith("https://"):
    raise RuntimeError(
        f"URL pública inválida: {dashboard_url!r}"
    )


# -------------------------------------------------------------------------
# 8. Resultado
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("✅ CÉLULA 2 CONCLUÍDA")
print("✅ VERSÃO: 2026-08-25.3")
print(f"✅ QUERY: {QUERY}")
print(f"🌐 LANDING: {dashboard_url}")
print("=" * 80)

CÉLULA 2 — OPERAÇÃO + DASHBOARD
VERSÃO: 2026-08-25.3
$ git pull --ff-only origin main
$ /usr/bin/python3 scripts/run_colab.py --mode setup
$ /usr/bin/python3 scripts/run_colab.py --mode collect --query manual reconciliation spreadsheet operations --limit 30 --minimum-score 0

[DASHBOARD] Iniciando dashboard pública...
$ /usr/bin/python3 scripts/run_colab.py --mode dashboard --dashboard-wait-seconds 90

✅ CÉLULA 2 CONCLUÍDA
✅ VERSÃO: 2026-08-25.3
✅ QUERY: manual reconciliation spreadsheet operations
🌐 LANDING: https://mom-perth-mba-perfume.trycloudflare.com


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("/content/opportunity-radar/data/opportunity_radar_operational.db")

if not db_path.exists():
    raise FileNotFoundError(f"Banco não encontrado: {db_path}")

conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)

print("TABELAS:")
display(tables)

for table in tables["name"]:
    try:
        df = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)
        print(f"\n{'=' * 80}")
        print(f"{table}: {len(df)} registros")
        print(f"{'=' * 80}")
        display(df.tail(10))
    except Exception as e:
        print(f"[ERRO] {table}: {e}")

conn.close()